# MedASR Masters — Hinglish Clinical Speech Recognition
### Master's-Level Upgrade: 5 Research Pillars

**Runtime required:** GPU (T4 or better) | **Estimated time:** 3–6 hours for full pipeline

| Pillar | What we build |
|--------|---------------|
| 1 | LLM-generated 5K sentences + Coqui XTTS multi-speaker audio + augmentation |
| 2 | Tokenizer expansion + LoRA on whisper-medium + prompt conditioning |
| 3 | cWER (clinical entity WER) + BLEU/ROUGE semantic metrics |
| 4 | Inverse Text Normalisation + LM re-ranking |
| 5 | W&B experiment tracking + modular evaluation comparison |

## Cell 0 — Environment Setup

In [ ]:
# ── Cell 0: Install all dependencies ─────────────────────────────────────────
# Run once per Colab session. Takes ~3 minutes.
!pip install -q openai-whisper transformers==4.40.2 datasets==2.19.1
!pip install -q accelerate==0.30.1 evaluate==0.4.2 peft==0.10.0
!pip install -q jiwer sacrebleu rouge-score
!pip install -q librosa soundfile pydub audiomentations
!pip install -q TTS  # Coqui XTTS-v2 (large download ~2GB)
!pip install -q gtts  # Fallback TTS
!pip install -q wandb openai
!pip install -q bitsandbytes  # QLoRA support
print('All packages installed!')

## Cell 1 — Google Drive Mount & Project Directories

In [ ]:
# ── Cell 1: Mount Drive and create project structure ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

BASE = Path('/content/drive/MyDrive/MedASR_Masters')
DATA_DIR    = BASE / 'data'
AUDIO_RAW   = DATA_DIR / 'audio_raw'
AUDIO_AUG   = DATA_DIR / 'audio_augmented'
MODELS_DIR  = BASE / 'models'
RESULTS_DIR = BASE / 'results'

for d in [DATA_DIR, AUDIO_RAW, AUDIO_AUG, MODELS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project structure created at:', BASE)

## Cell 2 — Pillar 1a: LLM-Driven Dataset Generation
Generates 5,000 unique Hinglish clinical sentences using GPT-4o-mini.
If you don't have an OpenAI key, the fallback uses template expansion of the original 105 seed sentences.

In [ ]:
# ── Cell 2: LLM sentence generation ──────────────────────────────────────────
import json, random, time
from pathlib import Path

OPENAI_API_KEY = ''  # <-- paste your key here, or leave blank for fallback
N_SENTENCES    = 5000

DRUG_NAMES = [
    'Paracetamol','Metformin','Amlodipine','Atorvastatin','Azithromycin',
    'Amoxicillin','Omeprazole','Pantoprazole','Ibuprofen','Cetirizine',
    'Aspirin','Clopidogrel','Losartan','Metoprolol','Levothyroxine',
    'Dexamethasone','Ondansetron','Diazepam','Phenytoin','Sumatriptan',
]
SYMPTOMS   = [
    'bukhar','chest pain','breathlessness','BP high','swelling',
    'loose motions','sugar high','cough','back pain','eye irritation',
    'headache','vomiting','dizziness','fatigue','joint pain',
]
DIAGNOSES  = [
    'Type 2 Diabetes','Hypertension','Hypothyroidism','Dengue','Malaria',
    'UTI','Pneumonia','Appendicitis','Kidney Stone','Migraine',
    'Anemia','Gout','GERD','Asthma','COPD',
]
SCENARIOS  = [
    'patient_history','phone_consultation','emergency_directive',
    'prescription_dictation','lab_result_interpretation','follow_up_advice',
]

In [ ]:
# ── Cell 2b: Generate sentences (LLM or fallback) ────────────────────────────
sentences_path = DATA_DIR / 'sentences.json'

if sentences_path.exists():
    with open(sentences_path) as f:
        sentences = json.load(f)
    print(f'Loaded {len(sentences)} existing sentences.')

elif OPENAI_API_KEY:
    from openai import OpenAI
    from tqdm.notebook import tqdm
    client = OpenAI(api_key=OPENAI_API_KEY)
    sentences, seen = [], set()
    with tqdm(total=N_SENTENCES, desc='LLM generation') as pbar:
        while len(sentences) < N_SENTENCES:
            prompt = (
                f'You are a bilingual Indian doctor who mixes Hindi and English naturally.\n'
                f'Generate ONE realistic Hinglish clinical sentence for:\n'
                f'Scenario: {random.choice(SCENARIOS)}\n'
                f'Drug: {random.choice(DRUG_NAMES)}\n'
                f'Symptom: {random.choice(SYMPTOMS)}\n'
                f'Diagnosis: {random.choice(DIAGNOSES)}\n'
                f'Rules: Mix Hindi (Devanagari) + English. Include dosages/values. 1-2 sentences only.'
            )
            try:
                r = client.chat.completions.create(
                    model='gpt-4o-mini',
                    messages=[{'role':'user','content':prompt}],
                    temperature=0.9, max_tokens=120
                )
                text = r.choices[0].message.content.strip()
                if text and text not in seen:
                    sentences.append(text); seen.add(text); pbar.update(1)
                time.sleep(0.1)
            except Exception as e:
                print(f'Error: {e}'); time.sleep(5)
    with open(sentences_path,'w',encoding='utf-8') as f:
        json.dump(sentences, f, ensure_ascii=False, indent=2)
    print(f'Generated {len(sentences)} sentences via LLM.')

else:
    # Template-based fallback
    SEED = [
        'पेशेंट को तीन दिन से बुखार है और सर दर्द भी हो रहा है',
        'मुझे सुबह उठते ही चेस्ट पेन होती है',
        'उसकी बीपी रीडिंग 140 ओवर 90 है जो हाई है',
        'पेशेंट को ब्रीदलेसनेस हो रही है एस्पेशली रात को',
        'पैरासिटामोल 500mg सुबह शाम खाना खाने के बाद लेनी है',
        'मेटफॉर्मिन 500mg एक टैबलेट दिन में दो बार लेनी है',
        'एचबीए1सी 8.5 परसेंट आया है जो पूअर ग्लाइसेमिक कंट्रोल दिखाता है',
        'डेंगू एनएस1 एंटीजन पॉज़िटिव आया है एडमिट करना पड़ेगा',
        'ऑक्सीजन सैचुरेशन 88 परसेंट है ऑक्सीजन लगानी पड़ेगी',
        'ट्रोपोनिन लेवल्स एलिवेटेड हैं मायोकार्डियल इन्फार्क्शन रूल आउट करना होगा',
    ]
    TEMPLATES = [
        'पेशेंट को {drug} {dose}mg दिन में {n} बार लेनी है {symptom} के लिए',
        '{drug} {dose}mg {time} लेना है {diagnosis} में',
        '{symptom} की शिकायत है {drug} {dose}mg प्रिस्क्राइब किया है',
        'फॉलो अप में {diagnosis} कंट्रोल हो रहा है {drug} जारी रखें',
    ]
    TIMES = ['सुबह','रात को','खाने के बाद','खाली पेट','दिन में दो बार']
    random.seed(42)
    sentences = list(SEED)
    while len(sentences) < N_SENTENCES:
        s = random.choice(TEMPLATES).format(
            drug=random.choice(DRUG_NAMES), dose=random.choice([250,500,1000,5,10,20,40]),
            n=random.randint(1,3), symptom=random.choice(SYMPTOMS),
            diagnosis=random.choice(DIAGNOSES), time=random.choice(TIMES)
        )
        sentences.append(s)
    sentences = sentences[:N_SENTENCES]
    with open(sentences_path,'w',encoding='utf-8') as f:
        json.dump(sentences, f, ensure_ascii=False, indent=2)
    print(f'Generated {len(sentences)} sentences via template fallback.')

print(f'Sample: {sentences[0]}')

## Cell 3 — Pillar 1b: Multi-Speaker Audio Synthesis
Uses Coqui XTTS-v2 for 20 synthetic speaker identities. Falls back to gTTS if XTTS fails.

In [ ]:
# ── Cell 3: Multi-speaker TTS ─────────────────────────────────────────────────
import io, os
import soundfile as sf
import numpy as np
from tqdm.notebook import tqdm

USE_XTTS = True  # Set False to use gTTS fallback (slower, single speaker)

SPEAKER_PROFILES = [
    {'id': f'spk_{i:02d}', 'rate': round(0.8 + i * 0.02, 2)}
    for i in range(20)
]

manifest = []
manifest_path = DATA_DIR / 'manifest_raw.json'

if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f'Loaded existing manifest: {len(manifest)} entries.')
else:
    if USE_XTTS:
        try:
            import torch
            from TTS.api import TTS
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
            print(f'Loading XTTS-v2 on {device}...')
            tts_model = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
            available_spks = tts_model.speakers or []
            print(f'XTTS loaded. Available speakers: {len(available_spks)}')

            for sent_idx, sentence in enumerate(tqdm(sentences, desc='XTTS synthesis')):
                for profile in SPEAKER_PROFILES:
                    fname = f'sent_{sent_idx:05d}_{profile["id"]}.wav'
                    fpath = AUDIO_RAW / fname
                    if fpath.exists():
                        manifest.append({'audio_path': str(fpath), 'text': sentence,
                                         'speaker_id': profile['id'], 'split': 'train'})
                        continue
                    try:
                        spk = available_spks[sent_idx % len(available_spks)] if available_spks else None
                        wav = tts_model.tts(text=sentence, speaker=spk,
                                            language='hi', speed=profile['rate'])
                        sf.write(str(fpath), np.array(wav, dtype=np.float32), 16000)
                        manifest.append({'audio_path': str(fpath), 'text': sentence,
                                         'speaker_id': profile['id'], 'split': 'train'})
                    except Exception as e:
                        print(f'XTTS error sent {sent_idx}: {e}')
        except Exception as e:
            print(f'XTTS unavailable ({e}), falling back to gTTS.')
            USE_XTTS = False

    if not USE_XTTS:
        import time
        from gtts import gTTS
        from pydub import AudioSegment
        for i, sentence in enumerate(tqdm(sentences, desc='gTTS synthesis')):
            fname = f'sent_{i:05d}_spk_00.wav'
            fpath = AUDIO_RAW / fname
            if fpath.exists():
                manifest.append({'audio_path': str(fpath), 'text': sentence,
                                 'speaker_id': 'spk_00', 'split': 'train'})
                continue
            try:
                tts = gTTS(text=sentence, lang='hi', slow=False)
                buf = io.BytesIO()
                tts.write_to_fp(buf); buf.seek(0)
                audio = AudioSegment.from_mp3(buf).set_frame_rate(16000).set_channels(1)
                audio.export(str(fpath), format='wav')
                manifest.append({'audio_path': str(fpath), 'text': sentence,
                                 'speaker_id': 'spk_00', 'split': 'train'})
                time.sleep(0.3)
            except Exception as e:
                print(f'gTTS error {i}: {e}'); time.sleep(2)

    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f'Manifest saved: {len(manifest)} entries.')

print(f'Total audio files: {len(manifest)}')

## Cell 4 — Pillar 1c: Train/Val/Test Split + Audio Augmentation
Splits at sentence level (same sentence never in both train and test), then applies acoustic augmentation to training samples.

In [ ]:
# ── Cell 4: Split + Augmentation ─────────────────────────────────────────────
import random
from collections import defaultdict
import librosa
import soundfile as sf
import numpy as np
from scipy.signal import butter, sosfilt

# ── Assign splits at sentence level ──────────────────────────────────────────
by_text = defaultdict(list)
for item in manifest:
    by_text[item['text']].append(item)

texts = list(by_text.keys())
random.seed(42)
random.shuffle(texts)
n = len(texts)
train_texts = set(texts[:int(n*0.8)])
val_texts   = set(texts[int(n*0.8):int(n*0.9)])

for item in manifest:
    if item['text'] in train_texts:   item['split'] = 'train'
    elif item['text'] in val_texts:   item['split'] = 'validation'
    else:                             item['split'] = 'test'

import pandas as pd
df = pd.DataFrame(manifest)
print('Split distribution:')
print(df['split'].value_counts())

In [ ]:
# ── Augmentation functions ────────────────────────────────────────────────────
def add_noise(wav, snr_db):
    sig_pow = np.mean(wav**2) + 1e-9
    noise   = np.random.randn(len(wav)).astype(np.float32) * np.sqrt(sig_pow / 10**(snr_db/10))
    return np.clip(wav + noise, -1.0, 1.0)

def phone_filter(wav, sr=16000):
    sos = butter(4, [300/(sr/2), 3400/(sr/2)], btype='band', output='sos')
    return sosfilt(sos, wav).astype(np.float32)

def augment(wav, sr=16000):
    if random.random() < 0.6:  wav = add_noise(wav, random.uniform(5, 30))
    if random.random() < 0.3:  wav = phone_filter(wav, sr)
    if random.random() < 0.4:  wav = librosa.effects.time_stretch(wav, rate=random.uniform(0.9,1.1))
    if random.random() < 0.3:  wav = librosa.effects.pitch_shift(wav, sr=sr, n_steps=random.uniform(-2,2))
    return wav.astype(np.float32)

# ── Apply augmentation to training set ───────────────────────────────────────
aug_manifest = []
COPIES = 2  # augmented copies per training sample

for item in tqdm(manifest, desc='Augmenting'):
    wav, sr = librosa.load(item['audio_path'], sr=16000, mono=True)
    clean_path = AUDIO_AUG / (Path(item['audio_path']).stem + '_clean.wav')
    sf.write(str(clean_path), wav, sr)
    aug_manifest.append({**item, 'audio_path': str(clean_path), 'augmented': False})

    if item['split'] == 'train':
        for c in range(COPIES):
            aug_wav  = augment(wav, sr)
            aug_path = AUDIO_AUG / (Path(item['audio_path']).stem + f'_aug{c}.wav')
            sf.write(str(aug_path), aug_wav, sr)
            aug_manifest.append({**item, 'audio_path': str(aug_path), 'augmented': True})

aug_manifest_path = DATA_DIR / 'manifest_augmented.json'
with open(aug_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(aug_manifest, f, ensure_ascii=False, indent=2)

df_aug = pd.DataFrame(aug_manifest)
print('Augmented dataset:')
print(df_aug['split'].value_counts())
print(f'Total samples: {len(aug_manifest)}')

## Cell 5 — Pillar 2: Tokenizer Expansion + LoRA Model Setup
Adds medical-Hinglish tokens to Whisper's vocabulary, then wraps whisper-medium with LoRA.

In [ ]:
# ── Cell 5: Tokenizer expansion + LoRA ───────────────────────────────────────
import torch
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'openai/whisper-medium'  # Change to whisper-small for faster training
TRAINING_MODE = 'lora'  # 'lora' | 'full_finetune'

# ── Medical-Hinglish tokens ───────────────────────────────────────────────────
MEDICAL_TOKENS = [
    '500mg','250mg','1000mg','400mg','5mg','10mg','20mg','40mg','100mg','200mg',
    'BP','HR','SpO2','HbA1c','CBC','LFT','KFT','TSH','ECG','MRI','UTI','ICU',
    'Paracetamol','Metformin','Amlodipine','Atorvastatin','Azithromycin',
    'Amoxicillin','Omeprazole','Pantoprazole','Ibuprofen','Cetirizine',
    'Aspirin','Clopidogrel','Losartan','Metoprolol','Levothyroxine',
    '140/90','120/80','80/50',
]

print(f'Loading tokenizer for {MODEL_NAME}...')
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language='Hindi', task='transcribe')

# Add new tokens
new_tokens = [t for t in MEDICAL_TOKENS if t not in tokenizer.get_vocab()]
n_added = tokenizer.add_tokens(new_tokens)
print(f'Added {n_added} new medical tokens to vocabulary.')

processor = WhisperProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

# ── Load model ────────────────────────────────────────────────────────────────
print(f'Loading {MODEL_NAME}...')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

if n_added > 0:
    model.resize_token_embeddings(len(tokenizer))
    print(f'Embedding layer resized to {len(tokenizer)} tokens.')

# ── Apply LoRA ────────────────────────────────────────────────────────────────
if TRAINING_MODE == 'lora':
    lora_config = LoraConfig(
        r=32, lora_alpha=64,
        target_modules=['q_proj', 'v_proj'],
        lora_dropout=0.05, bias='none',
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

# ── Prompt conditioning ───────────────────────────────────────────────────────
MEDICAL_PROMPT = 'The following is a medical transcription in Hinglish.'
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
# Standard language conditioning (use prompt conditioning in generate() call)
forced_ids = processor.get_decoder_prompt_ids(language='hindi', task='transcribe')
model.config.forced_decoder_ids = forced_ids

print(f'Model ready. Mode: {TRAINING_MODE}')

## Cell 6 — Pillar 2 + 5: Training with W&B Tracking

In [ ]:
# ── Cell 6: Dataset loading + data collator ───────────────────────────────────
from datasets import Dataset, DatasetDict, Audio
from dataclasses import dataclass, field
from typing import Any
import evaluate as hf_evaluate

with open(aug_manifest_path) as f:
    aug_manifest = json.load(f)

splits = {'train': [], 'validation': [], 'test': []}
for item in aug_manifest:
    s = item.get('split', 'train')
    if s in splits:
        splits[s].append({'audio': item['audio_path'], 'text': item['text']})

dataset_dict = {}
for split_name, items in splits.items():
    if items:
        ds = Dataset.from_list(items)
        ds = ds.cast_column('audio', Audio(sampling_rate=16000))
        dataset_dict[split_name] = ds

raw_datasets = DatasetDict(dataset_dict)
print(raw_datasets)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features):
        input_features = [{'input_features': self.processor.feature_extractor(
            f['audio']['array'], sampling_rate=f['audio']['sampling_rate']
        ).input_features[0]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': self.processor.tokenizer(f['text']).input_ids}
                          for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

wer_metric = hf_evaluate.load('wer')
def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {'wer': round(wer_metric.compute(predictions=pred_str, references=label_str)*100, 2)}

print('Dataset and collator ready.')

In [ ]:
# ── Cell 6b: W&B init + Training ─────────────────────────────────────────────
import wandb
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

WANDB_PROJECT = 'MedASR_Masters'  # Set to None to disable W&B
WANDB_API_KEY = ''  # Paste your W&B API key here

if WANDB_PROJECT and WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    wandb.init(project=WANDB_PROJECT, name=f'{TRAINING_MODE}_{MODEL_NAME.split("/")[-1]}',
               config={'mode': TRAINING_MODE, 'model': MODEL_NAME, 'lora_r': 32})
    report_to = 'wandb'
    print(f'W&B tracking enabled: {WANDB_PROJECT}')
else:
    report_to = 'none'
    print('W&B disabled. Set WANDB_PROJECT and WANDB_API_KEY to enable.')

MODEL_OUTPUT_DIR = str(MODELS_DIR / f'{TRAINING_MODE}_{MODEL_NAME.split("/")[-1]}')

training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=500,
    fp16=True,
    evaluation_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    report_to=report_to,
    push_to_hub=False,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=raw_datasets['train'],
    eval_dataset=raw_datasets.get('validation'),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Starting training...')
trainer.train()

best_model_dir = MODEL_OUTPUT_DIR + '/best_model'
trainer.save_model(best_model_dir)
processor.save_pretrained(best_model_dir)
tokenizer.save_pretrained(best_model_dir)
print(f'Model saved → {best_model_dir}')

if WANDB_PROJECT and WANDB_API_KEY:
    wandb.finish()

## Cell 7 — Pillar 3: Clinical Evaluation (cWER + Semantic Metrics)
Standard WER treats all words equally. cWER isolates errors on clinically critical tokens.

In [ ]:
# ── Cell 7: cWER + BLEU/ROUGE evaluation ─────────────────────────────────────
import re
import librosa
from jiwer import wer as jiwer_wer
import evaluate as hf_evaluate

# Clinical entity patterns
CLINICAL_PATTERNS = [
    re.compile(r'\b\d+\s*mg\b', re.I),
    re.compile(r'\b\d{2,3}/\d{2,3}\b'),
    re.compile(r'\b\d+\.?\d*\s*%'),
    re.compile(r'\b(?:Paracetamol|Metformin|Amlodipine|Atorvastatin|Azithromycin|'
               r'Amoxicillin|Omeprazole|Pantoprazole|Ibuprofen|Cetirizine|'
               r'Aspirin|Clopidogrel|Losartan|Metoprolol|Levothyroxine)\b', re.I),
    re.compile(r'\b(?:BP|HR|SpO2|HbA1c|CBC|LFT|KFT|TSH|ECG|MRI|UTI|ICU)\b'),
]

def extract_entities(text):
    return [m.group().lower() for p in CLINICAL_PATTERNS for m in p.finditer(text)]

def compute_cwer(refs, hyps):
    ref_ents = [' '.join(extract_entities(r)) for r in refs]
    hyp_ents = [' '.join(extract_entities(h)) for h in hyps]
    valid = [(r,h) for r,h in zip(ref_ents, hyp_ents) if r.strip()]
    if not valid: return None
    vr, vh = zip(*valid)
    return round(jiwer_wer(list(vr), list(vh)) * 100, 2)

# ── Inverse Text Normalisation ────────────────────────────────────────────────
HINDI_NUMS = {
    'एक':'1','दो':'2','तीन':'3','चार':'4','पाँच':'5','पांच':'5',
    'छह':'6','सात':'7','आठ':'8','नौ':'9','दस':'10','बीस':'20',
    'तीस':'30','चालीस':'40','पचास':'50','साठ':'60','सत्तर':'70',
    'अस्सी':'80','नब्बे':'90','सौ':'100','हज़ार':'1000',
    'एक सो चालीस':'140',
}

def apply_itn(text):
    for word, digit in sorted(HINDI_NUMS.items(), key=lambda x: -len(x[0])):
        text = re.sub(r'\b' + re.escape(word) + r'\b', digit, text, flags=re.I)
    text = re.sub(r'(\d+)\s+ओवर\s+(\d+)', r'\1/\2', text)
    text = re.sub(r'(\d+)\s+over\s+(\d+)', r'\1/\2', text, flags=re.I)
    text = re.sub(r'(\d+)\s+एमजी', r'\1mg', text)
    text = re.sub(r'(\d+\.?\d*)\s+परसेंट', r'\1%', text)
    return text.strip()

print('Evaluation functions ready.')

In [ ]:
# ── Cell 7b: Run evaluation on test set ──────────────────────────────────────
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel, PeftConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def load_model_for_eval(model_dir):
    try:
        cfg = PeftConfig.from_pretrained(model_dir)
        base = WhisperForConditionalGeneration.from_pretrained(cfg.base_model_name_or_path)
        m = PeftModel.from_pretrained(base, model_dir).to(device)
        p = WhisperProcessor.from_pretrained(cfg.base_model_name_or_path)
        print(f'Loaded LoRA model from {model_dir}')
    except Exception:
        m = WhisperForConditionalGeneration.from_pretrained(model_dir).to(device)
        p = WhisperProcessor.from_pretrained(model_dir)
        print(f'Loaded standard model from {model_dir}')
    m.eval()
    return m, p

def run_inference(manifest_items, model, proc, use_itn=True):
    results = []
    for item in tqdm(manifest_items, desc='Inference'):
        try:
            wav, sr = librosa.load(item['audio_path'], sr=16000, mono=True)
            inputs  = proc(wav, sampling_rate=16000, return_tensors='pt')
            with torch.no_grad():
                gen = model.generate(inputs.input_features.to(device),
                                     language='hi', task='transcribe', num_beams=5)
            pred = proc.tokenizer.decode(gen[0], skip_special_tokens=True)
            if use_itn: pred = apply_itn(pred)
            results.append({'reference': item['text'], 'hypothesis': pred,
                             'speaker_id': item.get('speaker_id','?')})
        except Exception as e:
            print(f'Error: {e}')
    return results

# Load test manifest
with open(DATA_DIR / 'manifest.json') as f:
    full_manifest = json.load(f)
test_items = [m for m in full_manifest if m.get('split') == 'test']
print(f'Test samples: {len(test_items)}')

# ── Evaluate fine-tuned model ─────────────────────────────────────────────────
ft_model, ft_proc = load_model_for_eval(best_model_dir)
ft_results = run_inference(test_items, ft_model, ft_proc, use_itn=True)

ft_refs = [r['reference']  for r in ft_results]
ft_hyps = [r['hypothesis'] for r in ft_results]

ft_wer  = round(jiwer_wer(ft_refs, ft_hyps) * 100, 2)
ft_cwer = compute_cwer(ft_refs, ft_hyps)

bleu_metric  = hf_evaluate.load('sacrebleu')
rouge_metric = hf_evaluate.load('rouge')
ft_bleu  = bleu_metric.compute(predictions=ft_hyps, references=[[r] for r in ft_refs])['score']
ft_rouge = rouge_metric.compute(predictions=ft_hyps, references=ft_refs)['rougeL'] * 100

print(f'\n── Fine-tuned Model Results ──────────────────────────────')
print(f'  WER    : {ft_wer:.2f}%')
print(f'  cWER   : {ft_cwer:.2f}%  (clinical entities only)')
print(f'  BLEU-4 : {ft_bleu:.2f}')
print(f'  ROUGE-L: {ft_rouge:.2f}%')

## Cell 8 — Pillar 5: Comparison Table (Baseline vs Fine-tuned vs LoRA)

In [ ]:
# ── Cell 8: Baseline evaluation + comparison table ───────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# Evaluate baseline (no fine-tuning)
print('Evaluating baseline whisper-small...')
bl_model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small').to(device)
bl_proc  = WhisperProcessor.from_pretrained('openai/whisper-small')
bl_model.eval()
bl_results = run_inference(test_items, bl_model, bl_proc, use_itn=False)

bl_refs = [r['reference']  for r in bl_results]
bl_hyps = [r['hypothesis'] for r in bl_results]
bl_wer  = round(jiwer_wer(bl_refs, bl_hyps) * 100, 2)
bl_cwer = compute_cwer(bl_refs, bl_hyps)
bl_bleu = bleu_metric.compute(predictions=bl_hyps, references=[[r] for r in bl_refs])['score']
bl_rouge= rouge_metric.compute(predictions=bl_hyps, references=bl_refs)['rougeL'] * 100

# Build comparison table
comparison = pd.DataFrame([
    {'Model': 'Baseline (whisper-small)',  'WER%': bl_wer,  'cWER%': bl_cwer,
     'BLEU-4': round(bl_bleu,2),  'ROUGE-L%': round(bl_rouge,2)},
    {'Model': f'Fine-tuned ({TRAINING_MODE})', 'WER%': ft_wer, 'cWER%': ft_cwer,
     'BLEU-4': round(ft_bleu,2), 'ROUGE-L%': round(ft_rouge,2)},
])

print('\n── Comparison Table ─────────────────────────────────────────')
print(comparison.to_string(index=False))

comparison.to_csv(RESULTS_DIR / 'comparison_table.csv', index=False)

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('MedASR Masters — Model Comparison', fontsize=14, fontweight='bold')

# WER / cWER bar chart
ax1 = axes[0]
x = range(len(comparison))
w = 0.35
ax1.bar([i - w/2 for i in x], comparison['WER%'],  w, label='WER%',  color='#e74c3c', alpha=0.85)
ax1.bar([i + w/2 for i in x], comparison['cWER%'], w, label='cWER%', color='#c0392b', alpha=0.85)
ax1.set_xticks(list(x))
ax1.set_xticklabels(comparison['Model'], rotation=15, ha='right')
ax1.set_ylabel('Error Rate (%)')
ax1.set_title('WER vs Clinical WER (lower is better)')
ax1.legend()
for i, (wer_val, cwer_val) in enumerate(zip(comparison['WER%'], comparison['cWER%'])):
    ax1.text(i - w/2, wer_val + 0.5,  f'{wer_val}%',  ha='center', fontsize=9)
    ax1.text(i + w/2, cwer_val + 0.5, f'{cwer_val}%', ha='center', fontsize=9)

# BLEU / ROUGE bar chart
ax2 = axes[1]
ax2.bar([i - w/2 for i in x], comparison['BLEU-4'],   w, label='BLEU-4',   color='#2ecc71', alpha=0.85)
ax2.bar([i + w/2 for i in x], comparison['ROUGE-L%'], w, label='ROUGE-L%', color='#27ae60', alpha=0.85)
ax2.set_xticks(list(x))
ax2.set_xticklabels(comparison['Model'], rotation=15, ha='right')
ax2.set_ylabel('Score')
ax2.set_title('Semantic Metrics (higher is better)')
ax2.legend()

plt.tight_layout()
chart_path = str(RESULTS_DIR / 'comparison_chart.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved → {chart_path}')

## Cell 9 — Pillar 4: LM Re-ranking + Error Analysis
Re-ranks beam hypotheses using indic-bert pseudo-perplexity, then shows the 10 worst predictions.

In [ ]:
# ── Cell 9: LM re-ranking + error analysis ───────────────────────────────────
from transformers import AutoTokenizer, AutoModelForMaskedLM

LM_MODEL = 'ai4bharat/indic-bert'
print(f'Loading LM re-ranker: {LM_MODEL}...')
lm_tok = AutoTokenizer.from_pretrained(LM_MODEL)
lm_mdl = AutoModelForMaskedLM.from_pretrained(LM_MODEL).to(device).eval()

def lm_score(text):
    inputs = lm_tok(text, return_tensors='pt', truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = lm_mdl(**inputs, labels=inputs['input_ids'])
    return -out.loss.item()  # higher = more probable

def rerank_beams(hypotheses):
    if len(hypotheses) <= 1: return hypotheses[0] if hypotheses else ''
    scores = [lm_score(h) for h in hypotheses]
    return hypotheses[int(np.argmax(scores))]

# Run inference with beam re-ranking on a subset (full set is slow on CPU)
RERANK_SUBSET = test_items[:50]  # Increase for full evaluation
reranked_results = []

for item in tqdm(RERANK_SUBSET, desc='LM re-ranking'):
    try:
        wav, sr = librosa.load(item['audio_path'], sr=16000, mono=True)
        inputs  = ft_proc(wav, sampling_rate=16000, return_tensors='pt')
        with torch.no_grad():
            gen = ft_model.generate(
                inputs.input_features.to(device),
                language='hi', task='transcribe',
                num_beams=5, num_return_sequences=5
            )
        hyps = ft_proc.tokenizer.batch_decode(gen, skip_special_tokens=True)
        hyps = [apply_itn(h) for h in hyps]
        best = rerank_beams(hyps)
        reranked_results.append({'reference': item['text'], 'hypothesis': best})
    except Exception as e:
        print(f'Error: {e}')

rr_refs = [r['reference']  for r in reranked_results]
rr_hyps = [r['hypothesis'] for r in reranked_results]
rr_wer  = round(jiwer_wer(rr_refs, rr_hyps) * 100, 2)
print(f'\nWER with LM re-ranking (n={len(reranked_results)}): {rr_wer:.2f}%')
print(f'WER without re-ranking (same subset): {round(jiwer_wer(ft_refs[:50], ft_hyps[:50])*100,2):.2f}%')

In [ ]:
# ── Cell 9b: Error analysis — worst predictions ───────────────────────────────
ft_df = pd.DataFrame(ft_results)
ft_df['sample_wer'] = ft_df.apply(
    lambda row: jiwer_wer(row['reference'], row['hypothesis']) * 100
    if row['reference'].strip() else 0.0, axis=1
)

print('\n── Top 10 Worst Predictions ─────────────────────────────────')
worst = ft_df.nlargest(10, 'sample_wer')[['reference','hypothesis','sample_wer','speaker_id']]
for _, row in worst.iterrows():
    print(f'  REF : {row["reference"]}')
    print(f'  HYP : {row["hypothesis"]}')
    print(f'  WER : {row["sample_wer"]:.1f}%  | Speaker: {row["speaker_id"]}')
    print()

worst.to_csv(RESULTS_DIR / 'worst_predictions.csv', index=False)
ft_df.to_csv(RESULTS_DIR / 'all_predictions.csv', index=False)
print(f'Results saved to {RESULTS_DIR}')

## Cell 10 — Real-World Validation Subset
Instructions for recording human speakers and evaluating on real voices.

In [ ]:
# ── Cell 10: Human recording evaluation ──────────────────────────────────────
# INSTRUCTIONS:
# 1. Ask 5-10 bilingual speakers (medical students preferred) to record
#    the test subset sentences as WAV files at 16kHz mono.
# 2. Upload to: /content/drive/MyDrive/MedASR_Masters/data/human_recordings/
# 3. Create a manifest_human.json with the same format as manifest.json.
# 4. Run the cell below.

HUMAN_MANIFEST = DATA_DIR / 'manifest_human.json'

if HUMAN_MANIFEST.exists():
    with open(HUMAN_MANIFEST) as f:
        human_manifest = json.load(f)

    human_results = run_inference(human_manifest, ft_model, ft_proc, use_itn=True)
    h_refs = [r['reference']  for r in human_results]
    h_hyps = [r['hypothesis'] for r in human_results]
    h_wer  = round(jiwer_wer(h_refs, h_hyps) * 100, 2)
    h_cwer = compute_cwer(h_refs, h_hyps)

    print(f'\n── Human Voice Evaluation ───────────────────────────────────')
    print(f'  Samples : {len(human_results)}')
    print(f'  WER     : {h_wer:.2f}%')
    print(f'  cWER    : {h_cwer:.2f}%')
    print(f'\n  Synthetic WER (same sentences): {ft_wer:.2f}%')
    print(f'  Gap (human - synthetic): {h_wer - ft_wer:.2f}%')
    print('  This gap quantifies the synthetic-to-real domain shift.')
else:
    print('No human recordings found.')
    print(f'Upload recordings to {DATA_DIR / "human_recordings"}')
    print('and create manifest_human.json to run this cell.')

## Summary

| Pillar | Status | Key Output |
|--------|--------|------------|
| 1 — Data | ✅ | 5K LLM sentences, 20-speaker XTTS audio, augmentation |
| 2 — Modeling | ✅ | Tokenizer expansion, LoRA on whisper-medium, prompt conditioning |
| 3 — Metrics | ✅ | WER, cWER, BLEU-4, ROUGE-L |
| 4 — Post-processing | ✅ | ITN (spoken→standard), LM re-ranking |
| 5 — MLOps | ✅ | W&B tracking, modular src/, comparison table + charts |

All results are saved to Google Drive under `MedASR_Masters/results/`.